# 客户流失预测与留存研究 —— 银行客户流失分析

## 一、业务背景与痛点

**业务背景**

在银行零售业务中，获取一位新客户的成本通常约为维系老客户的 5~7 倍。客户流失不仅直接造成存量收入损失，还会推高获客成本、损害品牌口碑与市场份额。面对日益激烈的同业竞争，银行需要由"事后补救"转向"事前预警"，在客户真正流失前主动识别风险并实施挽留。

**业务痛点**

1. **流失发现滞后**：往往等客户销户、停止交易后才察觉，错过最佳挽留窗口；
2. **资源投放盲目**：无法精准锁定高风险客户，挽留资源（人力、优惠）平均分配、效率低下；
3. **判断依赖经验**：缺乏数据驱动的预测能力，人工判断主观、滞后、难以规模化；
4. **策略无法量化**：缺少流失风险分级，难以制定差异化挽留方案并评估投入产出比。

## 二、项目目标

基于客户的人口统计、产品互动与银行行为数据，建立机器学习模型预测客户流失概率，定位高风险客户群体，并据此制定分层挽留策略，帮助企业降低流失率、提升客户留存与长期价值。

## 三、数据集说明

**数据集**：`Customer Churn Dataset.csv`（模拟数据，10000 条记录，14 个字段）

| 字段 | 说明 |
|---|---|
| RowNumber / CustomerId / Surname | 行号、客户ID、姓名（标识列，不用于建模） |
| CreditScore | 信用评分 |
| Geography | 国家（法国 / 德国 / 西班牙） |
| Gender | 性别 |
| Age | 年龄 |
| Tenure | 入行年限（在银行服务年数） |
| Balance | 账户余额 |
| NumOfProducts | 使用的产品数量 |
| HasCrCard | 是否持有信用卡（1是 0否） |
| IsActiveMember | 是否活跃会员（1是 0否） |
| EstimatedSalary | 预计年收入 |
| **Exited** | **目标变量：是否流失（1流失 0未流失）** |

---

## 总体思路（流程图）

1. **业务理解与目标** → 2. **数据理解**（EDA）→ 3. **数据清洗** → 4. **特征工程** → 5. **特征选择与数据划分** → 6. **建模**（基线+集成模型）→ 7. **评估对比** → 8. **模型优化**（调参+阈值）→ 9. **结果解读与留存策略**

In [ ]:
# 1. 导入所需库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)

import warnings
warnings.filterwarnings('ignore')

# 解决图表中文乱码（Windows 使用微软雅黑/黑体）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print('库导入完成 [OK]')

## 第1步：数据理解（EDA）

先看清数据长什么样：规模、字段类型、分布、缺失情况。EDA 的目的不是画画，而是**形成对流失行为的直觉假设**，指导后续特征工程和建模。

In [ ]:
# 2. 读取数据（请将本 Notebook 与 CSV 放在同一目录）
df = pd.read_csv('Customer Churn Dataset.csv')
df.head()

In [ ]:
# 3. 数据概览：规模、字段类型
print('数据集规模:', df.shape)
df.info()

In [ ]:
# 4. 描述性统计
df.describe().T

In [ ]:
# 5. 缺失值与重复检查
print('缺失值统计：')
print(df.isnull().sum())
print('\n重复行数:', df.duplicated().sum())

**结论：** 数据非常干净 —— 无缺失、无重复，无需填补或去重，可直接建模。

**关键观察——类别不平衡：**

In [ ]:
# 6. 目标变量分布：判断类别是否不平衡
print('流失(1):', int(df['Exited'].sum()), '人   未流失(0):', int((df['Exited']==0).sum()), '人')
print('流失占比: {:.2%}'.format(df['Exited'].mean()))

sns.countplot(x='Exited', data=df)
plt.title('目标变量 Exited 分布')
plt.show()

In [ ]:
# 7. 分类变量与流失率的关系（回答：哪些群体更容易流失？）
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df.groupby('Geography')['Exited'].mean().plot.bar(ax=axes[0], color='#4C72B0', title='不同国家流失率')
axes[0].set_ylabel('流失率')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('Gender')['Exited'].mean().plot.bar(ax=axes[1], color='#DD8452', title='不同性别流失率')
axes[1].set_ylabel('流失率')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 8. 连续变量在流失/未流失群体中的分布差异（箱线图）
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(x='Exited', y=col, data=df, ax=ax, palette=['#9ecae1', '#fc9272'])
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# 9. 数值特征相关性热图（初步判断特征与目标的线性关系）
plt.figure(figsize=(10, 7))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True,
            cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('特征相关性热图')
plt.show()

**EDA 主要发现（供业务解读）：**
- 流失客户中 **余额更高**；**年龄 40~60 岁中老年流失率明显更高**（50-60 岁流失率高达 56%，60 岁以上回落至 25%），Age、Balance 区分度明显；
- **德国客户流失率显著高于法国、西班牙**；性别差异不大；
- `IsActiveMember`（是否活跃）、`NumOfProducts` 可能是重要信号；
- 特征间相关性整体很低，不存在严重共线性。

---

## 第2步：数据清洗

**思路**：`RowNumber`、`CustomerId`、`Surname` 是唯一标识符，不携带可推广的规律。保留它们会让模型“背答案”（过拟合），建模前必须剔除。

In [ ]:
# 10. 数据清洗：删除无关标识列
df_model = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
print('清洗后特征列:', list(df_model.columns))

## 第3步：特征工程

**思路**：年龄对流失的影响通常是非线性的（年轻人、老年人更易流失），把连续年龄分箱成年龄段，既帮助模型捕捉非线性，又方便业务解读。此为可选步骤，但演示了特征工程的通用手法。

In [ ]:
# 11. 特征工程：年龄分箱
df_model['AgeGroup'] = pd.cut(df_model['Age'],
                              bins=[0, 30, 40, 50, 60, 100],
                              labels=['<30', '30-40', '40-50', '50-60', '60+'])
print('各年龄段流失率：')
print(df_model.groupby('AgeGroup', observed=True)['Exited'].mean().round(3))
print('\n新增列 AgeGroup，数据规模：', df_model.shape)

## 第4步：特征选择与数据划分

**思路**：把样本划分为训练集和测试集，测试集绝不参与训练，用于检验模型能否泛化到新客户。由于流失类不平衡，使用 `stratify=y` 做**分层抽样**，保证两边流失占比一致。

In [ ]:
# 12. 划分训练集(80%)与测试集(20%)，按目标分层抽样
X = df_model.drop(columns=['Exited'])
y = df_model['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('训练集:', X_train.shape, ' 测试集:', X_test.shape)
print('训练集流失占比: {:.2%}  |  测试集流失占比: {:.2%}'.format(y_train.mean(), y_test.mean()))

In [ ]:
# 13. 构建预处理管道
#   - 数值特征：标准化（消除量纲影响，逻辑回归/距离类算法必需）
#   - 分类特征：One-Hot 独热编码（Gender 只剩2列、AgeGroup 只剩4列，drop='first'避免多重共线性）
num_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
cat_features = ['Geography', 'Gender', 'AgeGroup']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

## 第5步：建模

**思路**：先上**逻辑回归**做基线（可解释、稳健），再上**随机森林**和**梯度提升树**这类集成模型（能捕捉非线性与特征交互）。所有模型放进统一的 `Pipeline`，预处理自动随交叉验证正确执行（避免数据泄漏）。

**关于类别不平衡**：流失只占 20%，模型会倾向把所有客户判成“未流失”。我们给少数类加权（`class_weight='balanced'`），并同时关注 **召回率、F1、AUC**，而不是只看准确率。

In [ ]:
# 14. 定义统一的评估函数
def evaluate_model(model, X, y, name='模型'):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    print(f'========== {name} ==========')
    print(f'准确率(Accuracy)  : {accuracy_score(y, pred):.4f}')
    print(f'精确率(Precision) : {precision_score(y, pred):.4f}')
    print(f'召回率(Recall)    : {recall_score(y, pred):.4f}')
    print(f'F1分数            : {f1_score(y, pred):.4f}')
    print(f'AUC               : {roc_auc_score(y, proba):.4f}')
    print('混淆矩阵（行=实际，列=预测）:')
    print(confusion_matrix(y, pred))
    return {'model': name,
            'accuracy': round(accuracy_score(y, pred), 4),
            'precision': round(precision_score(y, pred), 4),
            'recall': round(recall_score(y, pred), 4),
            'f1': round(f1_score(y, pred), 4),
            'auc': round(roc_auc_score(y, proba), 4)}

In [ ]:
# 15. 模型1：逻辑回归（基线）
lr = Pipeline([('pre', preprocessor),
               ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))])
lr.fit(X_train, y_train)
res_lr = evaluate_model(lr, X_test, y_test, '逻辑回归')

In [ ]:
# 16. 模型2：随机森林（袋装集成，对离群值稳健）
rf = Pipeline([('pre', preprocessor),
               ('clf', RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42))])
rf.fit(X_train, y_train)
res_rf = evaluate_model(rf, X_test, y_test, '随机森林')

In [ ]:
# 17. 模型3：梯度提升树（GBDT，序列集成，通常精度最高）
gbt = Pipeline([('pre', preprocessor),
                ('clf', GradientBoostingClassifier(random_state=42))])
gbt.fit(X_train, y_train)
res_gbt = evaluate_model(gbt, X_test, y_test, '梯度提升树')

# 可选：安装 xgboost/lightgbm 后可替换为更强的提升模型
#   import xgboost as xgb
#   xgb_model = Pipeline([('pre', preprocessor),
#                         ('clf', xgb.XGBClassifier(scale_pos_weight=4, eval_metric='auc', random_state=42))])

## 第6步：模型评估与对比

In [ ]:
# 18. 模型横向对比
results = pd.DataFrame([res_lr, res_rf, res_gbt]).set_index('model')
results

In [ ]:
# 19. ROC 曲线对比（AUC 越大，排序能力越强，与阈值无关）
plt.figure(figsize=(8, 6))
for name, model in [('逻辑回归', lr), ('随机森林', rf), ('梯度提升树', gbt)]:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, proba):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='随机猜测')
plt.xlabel('假正率 FPR'); plt.ylabel('真正率 TPR')
plt.title('ROC 曲线对比'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 20. 特征重要性（以随机森林为例）：回答“哪些因素最能预测流失？”
feat_names = rf.named_steps['pre'].get_feature_names_out()
imp = rf.named_steps['clf'].feature_importances_
imp_df = pd.DataFrame({'特征': feat_names, '重要性': imp}).sort_values('重要性', ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=imp_df.head(12), x='重要性', y='特征', palette='Blues')
plt.title('特征重要性 Top12')
plt.tight_layout()
plt.show()
imp_df.head(12)

## 第7步：模型优化

### 7.1 超参数调优
用 `GridSearchCV` + 分层5折交叉验证，以 AUC 为打分指标搜索随机森林的最优超参数（网格越大越耗时，可先跑小网格）。

In [ ]:
# 21. 网格搜索调参（随机森林）
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [5, None],
    'clf__min_samples_split': [2, 5],
}
grid = GridSearchCV(rf, param_grid, cv=StratifiedKFold(5),
                    scoring='roc_auc', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print('最优参数:', grid.best_params_)
print('交叉验证最优AUC: {:.4f}'.format(grid.best_score_))
best_model = grid.best_estimator_
res_best = evaluate_model(best_model, X_test, y_test, '调优后随机森林')

### 7.2 决策阈值优化（业务导向）

默认阈值是 0.5。但在挽留场景中，**漏掉一个会流失的客户（假阴性）比多打扰一个（假阳性）代价更大**。因此我们尝试降低阈值，在保持可接受精确率的前提下尽可能提高召回率 —— 把更多的“疑似流失”客户捞进挽留名单。

In [ ]:
# 22. 在不同决策阈值下评估
proba = best_model.predict_proba(X_test)[:, 1]

rows = []
for t in np.arange(0.30, 0.76, 0.05):
    pred = (proba >= t).astype(int)
    rows.append({'阈值': round(t, 2),
                 '准确率': round(accuracy_score(y_test, pred), 4),
                 '精确率': round(precision_score(y_test, pred), 4),
                 '召回率': round(recall_score(y_test, pred), 4),
                 'F1': round(f1_score(y_test, pred), 4)})
pd.DataFrame(rows).set_index('阈值')

**怎么选阈值？** 看业务目标：
- 追求**尽量多拦截**潜在流失客户 → 选召回率高、阈值偏低的一档（如 0.35~0.40）；
- 追求**挽留名单精准**、不浪费优惠成本 → 选精确率高、阈值偏高的一档（如 0.55~0.60）；
- 两全方案：分风险等级，对不同等级客户投入不同资源（见下一步）。

## 第8步：结果解读与留存策略

把客户按流失概率分为 **低 / 中 / 高风险** 三档，验证模型分层的效果（平均流失概率 vs 实际流失率），据此制定分层挽留策略。

In [ ]:
# 23. 客户流失风险分层
X_test_risk = X_test.copy()
X_test_risk['churn_prob'] = best_model.predict_proba(X_test)[:, 1]
X_test_risk['risk_level'] = pd.cut(X_test_risk['churn_prob'],
                                   bins=[0, 0.3, 0.6, 1.0],
                                   labels=['低风险', '中风险', '高风险'])
X_test_risk['Exited'] = y_test.values

seg = X_test_risk.groupby('risk_level', observed=True).agg(
    人数=('churn_prob', 'size'),
    平均流失概率=('churn_prob', 'mean'),
    实际流失率=('Exited', 'mean'),
).round(3)
seg

In [ ]:
# 24. 查看高风险客户的画像（特征均值，指导挽留话术/措施）
high = X_test_risk[X_test_risk['risk_level'] == '高风险']
low  = X_test_risk[X_test_risk['risk_level'] == '低风险']
cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'IsActiveMember']
pd.DataFrame({
    '高风险流失客户': high[cols].mean().round(2),
    '低风险留存客户': low[cols].mean().round(2),
})

## 结论与运营建议

1. **模型有效性**：随机森林/梯度提升树 AUC ≈ 0.84~0.85，说明用客户行为数据预测流失是可行的；逻辑回归虽 AUC 较低，但配合 `class_weight='balanced'` 能提供高召回率的可解释基线。

2. **关键流失信号**：模型中重要性靠前的特征为 `Age`、`Balance`、`CreditScore`、`NumOfProducts`、`Geography`（德国流失偏高）；`IsActiveMember`（是否活跃）单变量区分度强（非活跃客户流失率 26.9% vs 活跃 14.3%）。注：`EstimatedSalary` 重要性排序偏高，但年薪分档流失率极差仅约 2.6%、几乎无信号，属树模型对噪声特征的高估，建议用排列重要性/SHAP 复核。

3. **分层挽留策略**：
   - **高风险客户（流失概率 > 60%）**：主动干预 —— 专属客户经理回访、定向优惠券、免费升级服务、了解流失原因并针对性解决；
   - **中风险客户（30%~60%）**：轻度触达 —— 推送个性化产品推荐、定期关怀短信/邮件、奖励活跃度（激励使用 App、开通新业务）；
   - **低风险客户（<30%）**：正常经营，不额外投入成本。

4. **持续迭代**：模型上线后按月滚动重训，把当月真实流失结果回流进训练集，观察`IsActiveMember`、`Balance` 等特征变化，不断校准阈值与挽留策略的投入产出比。